# ANIMA-MECHANICUS — Fregate U-ALPHA : L'Auspex Cogitateur
> *L'Ame du Mouvement, Forgee par la Machine*

**Pipeline** : `.mp4 (humain reel)` → `[Gemini Chat]` → `[GVHMR SMPL]` → `[Retargeting R15]` → `.npz`

**Instructions (mode chat — sans cle API)** :
1. Cellule 1 — Installer les dependances (GVHMR, Torch CUDA 12.1, google-genai)
2. Cellule 1b — Telecharger SMPL_NEUTRAL.pkl depuis HuggingFace *(si manquant au Pre-Flight)*
3. Cellule 2 — Configurer les parametres pipeline *(GEMINI_API_KEY optionnel en mode chat)*
4. Cellule 3 — Pre-flight check (verifier que tout est pret)
5. Cellule 4 — Uploader la video `.mp4` source
6. Cellule 4b — Coller le JSON obtenu depuis Gemini Chat ([gemini.google.com](https://gemini.google.com))
7. Cellule 6 — Verifier le routing par segment
8. Cellule 7 — Lancer GVHMR → SMPL→R15 → exporter `.npz`
9. Cellule 8 — Telecharger les `.npz` (input de la Fregate U-GAMMA)

> **Prompt Gemini Chat** : voir `GEMINI_CHAT_METAPROMPT.md` dans le depot

**Runtime requis** : GPU T4 (Colab gratuit)

---


In [ ]:
# ══════ CELLULE 1 — INSTALLATION ══════
# GVHMR + Google Generative AI
# Duree estimee : 5-8 min sur Colab T4 (1 seule fois par session)
# Zero mmcv — Zero detectron2 — PyTorch CUDA 12.1

import os, subprocess, sys

def xpip(spec, extra_flags=""):
    cmd = f"pip install -q {extra_flags} {spec}"
    ret = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if ret.returncode != 0:
        stderr = ret.stderr
        if "setup.py egg_info" in stderr or "metadata-generation-failed" in stderr:
            print(f"  WARN setup.py legacy — retry --no-build-isolation")
            ret2 = subprocess.run(
                f"pip install -q --no-build-isolation {extra_flags} {spec}",
                shell=True, capture_output=True, text=True
            )
            if ret2.returncode != 0:
                print(f"  FAIL: {ret2.stderr[-300:].strip()}")
                return False
        else:
            print(f"  FAIL: {stderr[-300:].strip()}")
            return False
    return True

# ── [1/7] numpy — garder version pre-installee (scipy 1.16+ requiert numpy>=2.0) ──
print("[1/7] numpy...")
import subprocess as _sp, sys as _sys
_nv = _sp.run([_sys.executable, "-c", "import numpy; print(numpy.__version__)"],
              capture_output=True, text=True).stdout.strip()
# Ne pas downgrader : scipy >= 1.14 requiert numpy >= 2.0
# GVHMR fonctionne avec numpy 2.x
print(f"  OK numpy {_nv} (pre-installe — pas de downgrade)")

# ── [2/7] ffmpeg + aria2c ─────────────────────────────────────────────────
print("[2/7] ffmpeg + libgl1 + aria2...")
os.system("apt-get install -y ffmpeg libgl1 aria2 > /dev/null 2>&1")
print("  OK")

# ── [3/7] PyTorch CUDA 12.1 — compatible Colab T4 2026 ───────────────────
print("[3/7] PyTorch CUDA 12.1...")
_torch_check = subprocess.run(
    [sys.executable, "-c", "import torch; assert torch.cuda.is_available()"],
    capture_output=True
)
if _torch_check.returncode == 0:
    _tv = subprocess.run(
        [sys.executable, "-c", "import torch; print(torch.__version__)"],
        capture_output=True, text=True
    ).stdout.strip()
    print(f"  OK (pre-installe : {_tv})")
else:
    xpip("torch>=2.2.0 torchvision>=0.17.0",
         extra_flags="--index-url https://download.pytorch.org/whl/cu121")
    print("  OK")

# ── [4/7] google-genai ────────────────────────────────────────────────────
print("[4/7] google-genai...")
xpip("google-genai")
print("  OK")

# ── [5/7] GVHMR — clone + install ─────────────────────────────────────────
# GVHMR (SIGGRAPH Asia 2024) : https://github.com/zju3dv/GVHMR
# Zero mmcv, zero detectron2 : 2 commandes pip uniquement
print("[5/7] GVHMR...")
GVHMR_DIR = os.path.expanduser("~/GVHMR")
if not os.path.exists(GVHMR_DIR):
    os.system("git clone -q https://github.com/zju3dv/GVHMR.git ~/GVHMR")
else:
    os.system("cd ~/GVHMR && git pull -q")

# Etape A : filtrer pytorch3d de requirements.txt
# (wheel cp310 incompatible avec Python 3.12 — on installe le reste normalement)
_req_path = os.path.expanduser("~/GVHMR/requirements.txt")
_req_filtered = "/tmp/gvhmr_req_filtered.txt"
with open(_req_path) as _f:
    _lines = _f.readlines()
_kept = [l for l in _lines if "pytorch3d" not in l.lower()]
_skipped = [l.strip() for l in _lines if "pytorch3d" in l.lower()]
with open(_req_filtered, "w") as _f:
    _f.writelines(_kept)
if _skipped:
    print(f"  INFO : pytorch3d exclus (cp310 incompatible Python 3.12) : {_skipped[0][:60]}")

_r1 = subprocess.run(
    f"pip install -q -r {_req_filtered}",
    shell=True, capture_output=True, text=True
)
_r1_warns = [l for l in _r1.stderr.splitlines()
             if "error" in l.lower() or "fail" in l.lower()]
for _w in _r1_warns:
    print(f"  WARN req: {_w.strip()}")

# Etape B : installation GVHMR editable — TOUJOURS executer independamment
_r2 = subprocess.run(
    "pip install -q --no-deps -e ~/GVHMR",
    shell=True, capture_output=True, text=True
)
if _r2.returncode != 0:
    print(f"  WARN editable: {_r2.stderr[-200:].strip()}")
print("  OK")

# ── [5b/7] Packages supplementaires requis par GVHMR ─────────────────────
# Ces packages ne sont pas toujours installes par requirements.txt GVHMR
print("[5b] ultralytics + smplx + pytorch-lightning + colorlog...")
xpip("ultralytics smplx pytorch-lightning colorlog yacs chumpy cython_bbox lapx")
print("  OK")

# ── [5c/7] pytorch3d shim — remplacement pur PyTorch (zero compilation) ─────
# pytorch3d wheel cp310 incompatible Python 3.12. On cree un shim minimal
# qui fournit les fonctions utilisees par GVHMR (transforms seulement).
print("[5c] pytorch3d shim (pur PyTorch, zero CUDA compilation)...")
_shim_dir = "/tmp/pytorch3d_shim/pytorch3d"
os.makedirs(f"{_shim_dir}/transforms", exist_ok=True)
os.makedirs(f"{_shim_dir}/structures", exist_ok=True)
os.makedirs(f"{_shim_dir}/renderer", exist_ok=True)

with open(f"{_shim_dir}/__init__.py", "w") as _f:
    _f.write("# pytorch3d minimal shim for GVHMR\n")
with open(f"{_shim_dir}/structures/__init__.py", "w") as _f:
    _f.write("")
with open(f"{_shim_dir}/renderer/__init__.py", "w") as _f:
    _f.write("")

_transforms_code = '''
import torch, torch.nn.functional as F

def quaternion_to_matrix(q):
    r,i,j,k = torch.unbind(q,-1)
    s = 2.0/(q*q).sum(-1)
    o = torch.stack([1-s*(j*j+k*k),s*(i*j-k*r),s*(i*k+j*r),
                     s*(i*j+k*r),1-s*(i*i+k*k),s*(j*k-i*r),
                     s*(i*k-j*r),s*(j*k+i*r),1-s*(i*i+j*j)],-1)
    return o.reshape(q.shape[:-1]+(3,3))

def _sqrt_pos(x):
    return torch.where(x>0, torch.sqrt(torch.clamp(x,min=0)), torch.zeros_like(x))

def matrix_to_quaternion(m):
    b = m.shape[:-2]
    m00,m01,m02,m10,m11,m12,m20,m21,m22 = torch.unbind(m.reshape(b+(9,)),-1)
    q = _sqrt_pos(torch.stack([1+m00+m11+m22,1+m00-m11-m22,
                               1-m00+m11-m22,1-m00-m11+m22],-1)/4.0)
    w,x,y,z = torch.unbind(q,-1)
    r = torch.stack([w,torch.copysign(x,m21-m12),
                     torch.copysign(y,m02-m20),torch.copysign(z,m10-m01)],-1)
    return r/r.norm(dim=-1,keepdim=True).clamp(min=1e-8)

def axis_angle_to_quaternion(aa):
    ang = torch.norm(aa,p=2,dim=-1,keepdim=True).clamp(min=1e-6)
    h = ang*0.5
    s = torch.where(ang<1e-6, 0.5-(ang*ang)/48, torch.sin(h)/ang)
    return torch.cat([torch.cos(h),aa*s],-1)

def axis_angle_to_matrix(aa):
    return quaternion_to_matrix(axis_angle_to_quaternion(aa))

def quaternion_to_axis_angle(q):
    n = torch.norm(q[...,1:],p=2,dim=-1,keepdim=True)
    h = torch.atan2(n,q[...,:1])
    ang = 2*h
    s = torch.where(ang<1e-6, 0.5-(ang*ang)/48, torch.sin(h)/ang)
    return q[...,1:]/s.clamp(min=1e-8)

def matrix_to_axis_angle(m):
    return quaternion_to_axis_angle(matrix_to_quaternion(m))

def rotation_6d_to_matrix(d6):
    a1,a2 = d6[...,:3],d6[...,3:]
    b1 = F.normalize(a1,dim=-1)
    b2 = F.normalize(a2-(b1*a2).sum(-1,keepdim=True)*b1,dim=-1)
    b3 = torch.linalg.cross(b1,b2)
    return torch.stack((b1,b2,b3),dim=-2)

def matrix_to_rotation_6d(m):
    return m[...,:2,:].clone().reshape(m.shape[:-2]+(6,))

def standardize_quaternion(q):
    return torch.where(q[...,0:1]<0,-q,q)

def quaternion_raw_multiply(a,b):
    aw,ax,ay,az = torch.unbind(a,-1)
    bw,bx,by,bz = torch.unbind(b,-1)
    return torch.stack([aw*bw-ax*bx-ay*by-az*bz,
                        aw*bx+ax*bw+ay*bz-az*by,
                        aw*by-ax*bz+ay*bw+az*bx,
                        aw*bz+ax*by-ay*bx+az*bw],-1)

def quaternion_multiply(a,b):
    return standardize_quaternion(quaternion_raw_multiply(a,b))

def quaternion_invert(q):
    s = torch.tensor([1,-1,-1,-1],dtype=q.dtype,device=q.device)
    return q*s
'''
with open(f"{_shim_dir}/transforms/__init__.py", "w") as _f:
    _f.write(_transforms_code)

_setup = '''from setuptools import setup,find_packages
setup(name="pytorch3d",version="0.7.6.shim",packages=find_packages())'''
with open("/tmp/pytorch3d_shim/setup.py", "w") as _f:
    _f.write(_setup)

# pyproject.toml requis par pip 24+ (Python 3.12 Colab 2026)
_pyproject = '''[build-system]
requires = ["setuptools>=42"]
build-backend = "setuptools.backends.legacy:build"

[project]
name = "pytorch3d"
version = "0.7.6.shim"
'''
with open("/tmp/pytorch3d_shim/pyproject.toml", "w") as _f:
    _f.write(_pyproject)

_r_shim = subprocess.run(
    "pip install -q --no-deps /tmp/pytorch3d_shim",
    shell=True, capture_output=True, text=True
)
if _r_shim.returncode != 0:
    print(f"  WARN shim pip: {_r_shim.stderr[-200:].strip()}")
    print("  INFO : shim disponible via PYTHONPATH en fallback")
else:
    print("  OK (quaternion_to_matrix, axis_angle_to_matrix, rotation_6d ...)")

# ── [6/7] ANIMA-MECHANICUS ────────────────────────────────────────────────
print("[6/7] ANIMA-MECHANICUS...")
REPO_DIR = "/content/ANIMA-MECHANICUS"
if os.path.exists(REPO_DIR):
    os.system(f"cd {REPO_DIR} && git pull -q")
else:
    os.system(f"git clone -q https://github.com/kioka8877-ux/ANIMA-MECHANICUS.git {REPO_DIR}")
xpip("scipy opencv-python-headless 'scenedetect[opencv]'")
print("  OK")

# ── [7/7] Checkpoints GVHMR via aria2c (HuggingFace camenduru) ───────────
# Telechargement automatique — sans inscription — depuis HuggingFace
print("[7/7] Checkpoints GVHMR (aria2c depuis HuggingFace)...")
CKPT_BASE = os.path.expanduser("~/GVHMR/inputs/checkpoints")

# Format : (sous-dossier, nom_fichier, url_huggingface)
CHECKPOINTS = [
    # Checkpoint principal GVHMR (SIGGRAPH Asia 2024 release)
    ("gvhmr",
     "gvhmr_siga24_release.ckpt",
     "https://huggingface.co/camenduru/GVHMR/resolve/main/gvhmr/gvhmr_siga24_release.ckpt"),
    # HMR2.0a backbone
    ("hmr2",
     "epoch=10-step=25000.ckpt",
     "https://huggingface.co/camenduru/GVHMR/resolve/main/hmr2/epoch=10-step=25000.ckpt"),
    # ViTPose-H (pose 2D)
    ("vitpose",
     "vitpose-h-multi-coco.pth",
     "https://huggingface.co/camenduru/GVHMR/resolve/main/vitpose/vitpose-h-multi-coco.pth"),
    # YOLOv8x (detection personnes)
    ("yolo",
     "yolov8x.pt",
     "https://huggingface.co/camenduru/GVHMR/resolve/main/yolo/yolov8x.pt"),
    # Modele SMPL (body model)
    ("body_models/smpl",
     "SMPL_NEUTRAL.pkl",
     "https://huggingface.co/camenduru/SMPLer-X/resolve/main/SMPL_NEUTRAL.pkl"),
]

for subdir, fname, url in CHECKPOINTS:
    dst_dir = os.path.join(CKPT_BASE, subdir)
    os.makedirs(dst_dir, exist_ok=True)
    dst = os.path.join(dst_dir, fname)
    if not os.path.exists(dst):
        print(f"  Telechargement {fname}...")
        os.system(f"aria2c -c -x 16 -s 16 -k 1M -q '{url}' -d '{dst_dir}' -o '{fname}'")
        sz = os.path.getsize(dst)/1024/1024 if os.path.exists(dst) else 0
        print(f"  OK {fname} ({sz:.0f} MB)" if sz > 1 else f"  WARN {fname} ({sz:.1f} MB — verifier URL)")
    else:
        sz = os.path.getsize(dst)/1024/1024
        print(f"  OK {fname} deja present ({sz:.0f} MB)")

print("\n[U-ALPHA] Installation GVHMR terminee.")
print("Passer a la Cellule 2 — Configuration.")


In [ ]:
# ══════ CELLULE 1v — VERIFICATION POST-INSTALLATION ══════
# Lancer apres Cellule 1 pour confirmer que tous les modules sont importables.
# En cas de FAIL : relancer Cellule 1.

import importlib

print("[VERIF] Verification post-installation...\n")

CHECKS = [
    ("torch",        lambda m: f"PyTorch {m.__version__} | CUDA={__import__('torch').cuda.is_available()}"),
    ("torchvision",  lambda m: f"torchvision {m.__version__}"),
    ("numpy",        lambda m: f"numpy {m.__version__}"),
    ("scipy",        lambda m: f"scipy {m.__version__}"),
    ("cv2",          lambda m: f"cv2 {m.__version__}"),
    ("ultralytics",  lambda m: f"ultralytics {m.__version__}"),
    ("google.genai", lambda m: f"google-genai {m.__version__}"),
    ("einops",       lambda m: f"einops {m.__version__}"),
    ("smplx",        lambda m: f"smplx {getattr(m, '__version__', 'installed')}"),  # smplx n'expose pas __version__
]

ok_count = 0
fail_count = 0
for mod_name, fmt in CHECKS:
    try:
        mod = importlib.import_module(mod_name)
        print(f"  OK  {fmt(mod)}")
        ok_count += 1
    except ImportError as e:
        print(f"  FAIL  {mod_name} — {e}")
        fail_count += 1

# Verifier GVHMR importable
import os, sys
GVHMR_DIR = os.path.expanduser("~/GVHMR")
if os.path.exists(GVHMR_DIR):
    sys.path.insert(0, GVHMR_DIR)
    try:
        import gvhmr  # noqa
        print("  OK  gvhmr package")
        ok_count += 1
    except ImportError:
        # Verifier demo.py accessible (suffisant pour subprocess)
        demo = os.path.join(GVHMR_DIR, "tools", "demo", "demo.py")
        if os.path.exists(demo):
            print("  OK  GVHMR demo.py present")
            ok_count += 1
        else:
            print("  FAIL  GVHMR demo.py introuvable")
            fail_count += 1
else:
    print("  FAIL  GVHMR non clone")
    fail_count += 1

print(f"\n[VERIF] {ok_count} OK / {fail_count} FAIL")
if fail_count == 0:
    print("[VERIF] Toutes les dependances operationnelles — passer a la Cellule 2.")
else:
    print("[VERIF] Modules manquants — relancer la Cellule 1.")


In [ ]:
# ══════ CELLULE 1b — SMPL DEPUIS HUGGINGFACE ══════
# Telecharge SMPL_NEUTRAL.pkl depuis HuggingFace via aria2c (sans inscription)
# A executer si le Pre-Flight signale que SMPL_NEUTRAL.pkl est manquant
# Source : camenduru/SMPLer-X sur HuggingFace

import os, subprocess

SMPL_URL = "https://huggingface.co/camenduru/SMPLer-X/resolve/main/SMPL_NEUTRAL.pkl"
# Chemin attendu par GVHMR (dans inputs/checkpoints/body_models/smpl/)
DST_DIR  = os.path.expanduser("~/GVHMR/inputs/checkpoints/body_models/smpl")
DST      = os.path.join(DST_DIR, "SMPL_NEUTRAL.pkl")

os.makedirs(DST_DIR, exist_ok=True)

if os.path.exists(DST):
    sz = os.path.getsize(DST) / 1024 / 1024
    print(f"[SMPL] Deja present : {DST} ({sz:.1f} MB)")
    if sz < 1:
        print("  WARN : fichier trop petit — re-telechargement force")
        os.remove(DST)
    else:
        print("[SMPL] OK — rien a faire.")

if not os.path.exists(DST):
    print(f"[SMPL] Telechargement depuis HuggingFace...")
    ret = os.system(
        f"aria2c -c -x 16 -s 16 -k 1M '{SMPL_URL}' -d '{DST_DIR}' -o 'SMPL_NEUTRAL.pkl'"
    )
    if os.path.exists(DST):
        sz = os.path.getsize(DST) / 1024 / 1024
        if sz > 1:
            print(f"[SMPL] OK — {DST} ({sz:.1f} MB)")
        else:
            print(f"[SMPL] WARN : telechargement suspect ({sz:.1f} MB) — verifier l'URL")
            print(f"  URL : {SMPL_URL}")
    else:
        print("[SMPL] ECHEC : aria2c n'a pas produit le fichier")
        print(f"  Tentative alternative avec wget...")
        os.system(f"wget -q '{SMPL_URL}' -O '{DST}'")
        if os.path.exists(DST) and os.path.getsize(DST) > 1024*1024:
            print(f"[SMPL] OK via wget ({os.path.getsize(DST)/1024/1024:.1f} MB)")
        else:
            print("[SMPL] ECHEC. URL peut etre obsolete.")
            print("  Alternatives :")
            print("    1. Verifier camenduru/GVHMR sur HuggingFace pour le fichier SMPL_NEUTRAL.pkl")
            print("    2. S'inscrire sur https://smpl.is.tue.mpg.de/ et uploader le pkl manuellement")

print(f"\n[SMPL] Chemin attendu : {DST}")
print("[SMPL] Relancer la Cellule 3 (Pre-Flight Check) pour verification.")


In [ ]:
# ══════ CELLULE 2 — CONFIGURATION ══════

import os

# @title Parametres U-ALPHA

GEMINI_API_KEY      = ""       # @param {type:"string"} — obtenir sur https://aistudio.google.com
FPS_CIBLE           = 30       # @param [30, 60, 120] {type:"raw"}
LISSAGE             = "moyen" # @param ["faible", "moyen", "brutal"] {type:"string"}
ROOT_MOTION         = True     # @param {type:"boolean"}
QUALITY_THRESHOLD   = 0.6      # @param {type:"number"}
SMPL_MODEL_PATH     = os.path.expanduser("~/GVHMR/inputs/checkpoints/body_models/smpl/SMPL_NEUTRAL.pkl")

# Cache Gemini — evite de re-consommer du quota sur la meme video
# Laisser vide "" pour desactiver, ou garder le chemin par defaut
GEMINI_CACHE_DIR    = "/content/gemini_cache"  # @param {type:"string"}

GVHMR_DIR  = os.path.expanduser("~/GVHMR")
REPO_DIR   = "/content/ANIMA-MECHANICUS"
OUTPUT_DIR = "/content/alpha_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
if GEMINI_CACHE_DIR:
    os.makedirs(GEMINI_CACHE_DIR, exist_ok=True)

print(f"[CONFIG] FPS={FPS_CIBLE} | Lissage={LISSAGE} | RootMotion={ROOT_MOTION} | Seuil={QUALITY_THRESHOLD}")
print(f"[CONFIG] Cache Gemini : {GEMINI_CACHE_DIR if GEMINI_CACHE_DIR else 'desactive'}")
print("[U-ALPHA] Configuration sauvegardee — passer a la Cellule 3 (pre-flight check)")


In [ ]:
# ══════ CELLULE 3 — PRE-FLIGHT CHECK ══════
# Verifie que toutes les dependances sont operationnelles avant de lancer GVHMR

import os, sys, subprocess

errors  = []
warnings = []

print("[PRE-FLIGHT] Verification de l'environnement...\n")

# 1. GPU
gpu_check = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                             "--format=csv,noheader"], capture_output=True, text=True)
if gpu_check.returncode == 0:
    gpu_info = gpu_check.stdout.strip()
    print(f"  OK  GPU : {gpu_info}")
else:
    errors.append("GPU non detecte — activer le runtime GPU T4 dans Colab (Execution → Modifier le type d'execution)")

# 2. PyTorch + CUDA
try:
    import torch
    cuda_ok = torch.cuda.is_available()
    print(f"  OK  PyTorch {torch.__version__} | CUDA : {'oui' if cuda_ok else 'NON'}")
    if not cuda_ok:
        warnings.append("PyTorch sans CUDA — GVHMR sera tres lent (CPU only)")
except ImportError:
    errors.append("PyTorch non installe — relancer la Cellule 1")

# 3. google-genai
try:
    from google import genai as _genai_check
    print(f"  OK  google-genai {_genai_check.__version__}")
except ImportError:
    errors.append("google-genai non installe — relancer la Cellule 1")

# 4. Cle Gemini — OPTIONNELLE en mode chat (Cellule 4b)
if not GEMINI_API_KEY:
    print("  INFO Gemini API key absente — mode chat actif (Cellule 4b)")
    print("       Cle non requise si tu utilises Gemini Chat + Cellule 4b.")
else:
    try:
        from google import genai as _genai_val
        _client = _genai_val.Client(api_key=GEMINI_API_KEY)
        list(_client.models.list())
        print("  OK  Gemini API key valide")
    except Exception as e:
        warnings.append(f"Gemini API key invalide ou reseau KO : {e} — utilise la Cellule 4b a la place")

# 5. GVHMR
if not os.path.exists(GVHMR_DIR):
    errors.append(f"GVHMR introuvable : {GVHMR_DIR} — relancer Cellule 1")
else:
    demo = os.path.join(GVHMR_DIR, "tools", "demo", "demo.py")
    if os.path.exists(demo):
        print(f"  OK  GVHMR : {GVHMR_DIR}")
    else:
        errors.append(f"GVHMR incomplet (tools/demo/demo.py absent) — re-cloner : git clone https://github.com/zju3dv/GVHMR.git ~/GVHMR")

# 6. Modele SMPL
if not os.path.exists(SMPL_MODEL_PATH):
    errors.append(
        f"Modele SMPL introuvable : {SMPL_MODEL_PATH}\n"
        "  → Executer la Cellule 1b pour le telecharger automatiquement depuis HuggingFace"
    )
else:
    size_mb = os.path.getsize(SMPL_MODEL_PATH) / 1024 / 1024
    if size_mb < 1:
        errors.append(f"Modele SMPL trop petit ({size_mb:.1f} MB) — fichier corrompu — relancer Cellule 1b")
    else:
        print(f"  OK  Modele SMPL : {SMPL_MODEL_PATH} ({size_mb:.1f} MB)")

# 7. scipy / numpy / cv2
for pkg in ["numpy", "scipy", "cv2"]:
    try:
        mod = __import__(pkg)
        ver = getattr(mod, "__version__", "?")
        print(f"  OK  {pkg} {ver}")
    except ImportError:
        errors.append(f"{pkg} non installe — relancer Cellule 1")

# 8. ffmpeg
ffmpeg_check = subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True)
if ffmpeg_check.returncode == 0:
    version_line = ffmpeg_check.stdout.splitlines()[0]
    print(f"  OK  {version_line[:50]}")
else:
    errors.append("ffmpeg non installe — relancer Cellule 1")

# 9. Codebase ANIMA-MECHANICUS
extract_script = f"{REPO_DIR}/U-ALPHA/codebase/motus_extract.py"
if os.path.exists(extract_script):
    print(f"  OK  motus_extract.py present")
else:
    errors.append(f"motus_extract.py introuvable — relancer Cellule 1")

# ── Bilan ─────────────────────────────────────────────────────────────────
print()
if warnings:
    print("AVERTISSEMENTS :")
    for w in warnings:
        print(f"  [WARN] {w}")
    print()

if errors:
    print(f"PRE-FLIGHT ECHEC — {len(errors)} erreur(s) a corriger :")
    for i, e in enumerate(errors, 1):
        print(f"  [{i}] {e}")
    raise RuntimeError("Corriger les erreurs ci-dessus avant de continuer")
else:
    print("PRE-FLIGHT OK — Tous les systemes sont operationnels.")
    if not GEMINI_API_KEY:
        print("  Mode chat : utilise Cellule 4b pour injecter le JSON Gemini.")
    else:
        print("Passer a la Cellule 4 pour uploader la video.")


In [ ]:
# ══════ CELLULE 4 — UPLOAD VIDEO ══════
# Uploader la video .mp4 source (humain reel uniquement, max 60 secondes)

from google.colab import files as colab_files
import os, cv2

VIDEO_PATH = None

print("Option A — Upload direct depuis votre ordinateur :")
try:
    uploaded = colab_files.upload()
    for fname, data in uploaded.items():
        if fname.lower().endswith(".mp4"):
            VIDEO_PATH = f"/content/{fname}"
            with open(VIDEO_PATH, "wb") as f:
                f.write(data)
            break
except Exception:
    pass

if not VIDEO_PATH:
    VIDEO_PATH_MANUEL = ""  # @param {type:"string"}
    if VIDEO_PATH_MANUEL and os.path.exists(VIDEO_PATH_MANUEL):
        VIDEO_PATH = VIDEO_PATH_MANUEL

if not VIDEO_PATH or not os.path.exists(VIDEO_PATH):
    raise ValueError("[ERREUR] Aucune video .mp4 chargee")

cap = cv2.VideoCapture(VIDEO_PATH)
src_fps    = cap.get(cv2.CAP_PROP_FPS)
tot_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width      = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
duration   = tot_frames / src_fps if src_fps > 0 else 0
cap.release()
size_mb = os.path.getsize(VIDEO_PATH) / 1024 / 1024

print(f"  OK  {os.path.basename(VIDEO_PATH)}")
print(f"      {width}x{height} | {src_fps:.1f} FPS | {duration:.1f}s | {size_mb:.1f} MB")

if duration > 60:
    print(f"  [WARN] Duree {duration:.1f}s > 60s — pipeline prevu pour max 60 secondes")
if width < 480 or height < 480:
    print(f"  [WARN] Resolution faible ({width}x{height}) — qualite GVHMR potentiellement reduite")


In [ ]:
# ══════ CELLULE 4b — INJECTION JSON GEMINI (MODE CHAT) ══════
# Utilise cette cellule si tu as obtenu le JSON depuis gemini.google.com (chat)
# Apres validation, ALL_SEGMENTS est pret — passe directement a la Cellule 6.
# Cellule 5 (API) sera automatiquement ignoree.

import sys, os, hashlib
import json as _json
import ipywidgets as widgets
from IPython.display import display

sys.path.insert(0, f"{REPO_DIR}/U-ALPHA/codebase")

assert VIDEO_PATH and os.path.exists(VIDEO_PATH), "[ERREUR] Charger une video en Cellule 4 d'abord"

_ta = widgets.Textarea(
    placeholder='Colle ici le JSON complet fourni par Gemini Chat...',
    layout=widgets.Layout(width='100%', height='280px')
)
_btn = widgets.Button(description='Valider et injecter', button_style='success',
                      layout=widgets.Layout(width='200px'))
_out = widgets.Output()

def _on_inject(b):
    global ALL_SEGMENTS, GEMINI_JSON_INJECTED, GEMINI_DATA
    with _out:
        _out.clear_output()
        raw = _ta.value.strip()
        if not raw:
            print("[SKIP] Zone vide — rien a injecter")
            return
        try:
            data = _json.loads(raw)
        except _json.JSONDecodeError as e:
            print(f"[ERREUR] JSON invalide : {e}")
            return

        # Sauvegarder en cache (pour reutilisation future)
        cache_dir = GEMINI_CACHE_DIR or "/content/gemini_cache"
        os.makedirs(cache_dir, exist_ok=True)
        with open(VIDEO_PATH, "rb") as f:
            h = hashlib.md5(f.read(1024 * 1024)).hexdigest()[:10]
        stem = os.path.splitext(os.path.basename(VIDEO_PATH))[0]
        cache_path = os.path.join(cache_dir, f"gemini_cache_{stem}_{h}.json")
        with open(cache_path, "w", encoding="utf-8") as f:
            _json.dump(data, f, ensure_ascii=False, indent=2)

        # Filtrer les segments
        from motus_extract import filter_segments
        segs = filter_segments(data, QUALITY_THRESHOLD)

        # Injecter dans les globales
        GEMINI_DATA          = data
        ALL_SEGMENTS         = segs
        GEMINI_JSON_INJECTED = True

        # Rapport
        print(f"  JSON valide : {len(data['persons'])} personne(s) | "
              f"{data['video_duration_seconds']}s | qualite={data['qualite_globale']}")
        print()
        for p in data['persons']:
            print(f"  Personne {p['person_id']} :")
            for seg in p.get('segments_valides', []):
                modele = seg.get('modele_recommande', '?')
                extraction = seg.get('extraction_possible', '?')
                dur = seg['end_s'] - seg['start_s']
                print(f"    [{seg['start_s']}s -> {seg['end_s']}s ({dur:.1f}s)] "
                      f"{seg['corps_visible']} -> {modele} ({extraction})")
            for seg in p.get('segments_exclus', []):
                print(f"    [{seg['start_s']}s -> {seg['end_s']}s] EXCLU : {seg['raison']}")
        print()
        print(f"  ALL_SEGMENTS : {len(segs)} segment(s) retenus (seuil >= {QUALITY_THRESHOLD})")
        print(f"  Cache sauvegarde : {cache_path}")
        print()
        print("  OK — Passe directement a la Cellule 6 — Cellule 5 sera ignoree automatiquement")

_btn.on_click(_on_inject)
display(widgets.VBox([_ta, _btn, _out]))


In [ ]:
# ══════ CELLULE 5 — ANALYSE GEMINI API (AUTO) ══════
# A utiliser UNIQUEMENT si tu n'as PAS utilise la Cellule 4b.
# Si ALL_SEGMENTS est deja defini (via Cellule 4b), cette cellule se skipe.

import sys, os
sys.path.insert(0, f"{REPO_DIR}/U-ALPHA/codebase")
from motus_extract import analyze_video_gemini, print_gemini_report, filter_segments

# ── Garde : skip si JSON deja injecte via 4b ──────────────────────────
if globals().get('GEMINI_JSON_INJECTED'):
    print("[U-ALPHA][Cellule 5] JSON deja injecte via Cellule 4b — cellule ignoree")
    print(f"  ALL_SEGMENTS : {len(ALL_SEGMENTS)} segment(s) deja disponibles")
    print("  Passe a la Cellule 6.")
else:
    assert GEMINI_API_KEY, "[ERREUR] Configurer GEMINI_API_KEY en Cellule 2 (ou utiliser Cellule 4b)"
    assert VIDEO_PATH and os.path.exists(VIDEO_PATH), "[ERREUR] Charger une video en Cellule 4"

    print("[U-ALPHA] Analyse Gemini API (cascade auto)...")
    if GEMINI_CACHE_DIR:
        print(f"[U-ALPHA] Cache actif : {GEMINI_CACHE_DIR}")

    GEMINI_DATA = analyze_video_gemini(
        VIDEO_PATH,
        GEMINI_API_KEY,
        cache_dir=GEMINI_CACHE_DIR,
    )
    print_gemini_report(GEMINI_DATA)

    ALL_SEGMENTS = filter_segments(GEMINI_DATA, QUALITY_THRESHOLD)

    print(f"{'='*60}")
    print(f"[FILTRE] {len(ALL_SEGMENTS)} segment(s) retenus (seuil >= {QUALITY_THRESHOLD}) :")
    for s in ALL_SEGMENTS:
        dur = s['end_s'] - s['start_s']
        modele = s.get('modele_recommande', 'GVHMR')
        print(f"  P{s['person_id']} | {s['start_s']:.1f}s-{s['end_s']:.1f}s ({dur:.1f}s) | "
              f"{s['corps_visible']} | qualite={s['qualite_estimee']:.2f} | {modele}")
    print(f"{'='*60}")

    cam = GEMINI_DATA.get("camera", {})
    if cam.get("mouvement") == "agitee":
        print("[WARN] Camera agitee — root motion peut etre peu fiable")
    if cam.get("zoom_detecte"):
        print("[WARN] Zoom detecte — root motion peut etre fausse")
    if not ALL_SEGMENTS:
        print("[ERREUR] Aucun segment valide — verifier que la video contient un humain clairement visible")


In [ ]:
# ══════ CELLULE 6 — VALIDATION DES SEGMENTS ══════

# === Ajustements manuels optionnels ===
# SEGMENTS_TO_PROCESS = [s for s in ALL_SEGMENTS if s['person_id'] == 1]
# SEGMENTS_TO_PROCESS = [s for s in ALL_SEGMENTS if s.get('modele_recommande') in ('WHAM','GVHMR')]
# SEGMENTS_TO_PROCESS = [s for s in ALL_SEGMENTS if s['start_s'] >= 5.0]
# SEGMENTS_TO_PROCESS = [s for s in ALL_SEGMENTS if s['corps_visible'] == 'complet']

SEGMENTS_TO_PROCESS = ALL_SEGMENTS

# ── Routing par modele ────────────────────────────────────────────────
# _skip_gvhmr = True sur les segments DECA (non implementes)
# _mask_lower_body = True sur les segments FrankMocap_upper (haut du corps)
_gvhmr_segs  = [s for s in SEGMENTS_TO_PROCESS if not s.get('_skip_gvhmr')]
_gvhmr_full  = [s for s in _gvhmr_segs if s.get('modele_recommande', 'GVHMR') in ('WHAM', 'GVHMR')]
_frank_segs  = [s for s in _gvhmr_segs if s.get('_mask_lower_body')]
_deca_segs   = [s for s in SEGMENTS_TO_PROCESS if s.get('_skip_gvhmr')]
_full_dur    = sum(s['end_s'] - s['start_s'] for s in _gvhmr_full)
_frank_dur   = sum(s['end_s'] - s['start_s'] for s in _frank_segs)

print(f"[U-ALPHA] Routing des {len(SEGMENTS_TO_PROCESS)} segment(s) valides :")
print()

MODELE_LABEL = {
    'WHAM':             '\u2705 GVHMR (legacy)',
    'GVHMR':            '\u2705 GVHMR         ',
    'FrankMocap_upper': '\u2139\ufe0f  GVHMR+masque  ',
    'DECA':             '\u26a0\ufe0f  DECA (ignore) ',
    'skip':             '\u23ed\ufe0f  SKIP          ',
}

for s in SEGMENTS_TO_PROCESS:
    dur = s['end_s'] - s['start_s']
    modele = s.get('modele_recommande', 'GVHMR')
    extraction = s.get('extraction_possible', '?')
    label = MODELE_LABEL.get(modele, modele)
    mask_tag = ' [joints inf. masques]' if s.get('_mask_lower_body') else ''
    skip_tag = ' [ignore]' if s.get('_skip_gvhmr') else ''
    print(f"  {label} | P{s['person_id']} {s['start_s']:.1f}s\u2192{s['end_s']:.1f}s "
          f"({dur:.1f}s) | {s['corps_visible']} | {extraction}{mask_tag}{skip_tag}")

print()
print(f"  Segments GVHMR corps complet        : {len(_gvhmr_full)} ({_full_dur:.1f}s)")
if _frank_segs:
    print(f"  Segments haut du corps (masquage)   : {len(_frank_segs)} ({_frank_dur:.1f}s)")
    print(f"    [INFO] Joints inferieurs → quaternion identite [1,0,0,0]")
    print(f"           Root motion conserve — seuls les bras/torse animes")
if _deca_segs:
    print(f"  Segments DECA (ignores)             : {len(_deca_segs)}")
print()

if not _gvhmr_segs:
    print("[WARN] Aucun segment GVHMR disponible.")
    if _deca_segs:
        print("       Tous les segments sont DECA — non encore implementes.")
    raise ValueError("Aucun segment GVHMR — ajuste la selection ci-dessus")
else:
    total_gvhmr = len(_gvhmr_segs)
    print(f"  OK — {total_gvhmr} segment(s) envoyes a GVHMR. Passer a la Cellule 7.")


In [ ]:
# ══════ CELLULE 6b — VERIFICATION DES DEPENDANCES ══════
# A lancer si Cellule 7 plante sur un ModuleNotFoundError.
# Verifie que tous les packages requis par GVHMR sont bien installes
# et installe les manquants sans re-executer toute la Cellule 1.

import sys, subprocess

# Tuple : (nom_import, commande_pip_ou_None)
# None = reinstallation manuelle requise (torch, cv2...)
REQUIRED = [
    ("einops",       "einops"),
    ("smplx",        "smplx"),
    ("ultralytics",  "ultralytics"),
    ("loguru",       "loguru"),
    ("joblib",       "joblib"),
    ("imageio",      "imageio[ffmpeg]"),
    ("tensorboard",  "tensorboard"),
    ("torch",        None),
    ("cv2",          None),
    ("scipy",        None),
    ("numpy",        None),
]

missing_to_install = []
already_ok         = []
skip_reinstall     = []

for import_name, pip_name in REQUIRED:
    try:
        __import__(import_name)
        already_ok.append(import_name)
    except ImportError:
        if pip_name is None:
            skip_reinstall.append(import_name)
        else:
            missing_to_install.append((import_name, pip_name))

print(f"[6b] OK ({len(already_ok)}) : {', '.join(already_ok)}")

if skip_reinstall:
    print(f"[6b] MANQUANTS (reinstallation manuelle) : {', '.join(skip_reinstall)}")
    if "cv2" in skip_reinstall:
        print("     cv2 manquant : !pip install -q opencv-python-headless")
    if "torch" in skip_reinstall:
        print("     torch manquant : relancer la Cellule 1 complete")

if missing_to_install:
    print(f"\n[6b] Installation des {len(missing_to_install)} package(s) manquant(s)...")
    for import_name, pip_spec in missing_to_install:
        parts = pip_spec.split()
        cmd = [sys.executable, "-m", "pip", "install", "-q"] + parts
        print(f"  {import_name} ...", end=" ", flush=True)
        ret = subprocess.run(cmd, capture_output=True)
        print("OK" if ret.returncode == 0 else f"FAIL\n    {ret.stderr.decode()[-200:].strip()}")
    print("\n[6b] Installation terminee — relancer Cellule 7.")
elif not skip_reinstall:
    print("\n[6b] Toutes les dependances presentes — Cellule 7 prete.")
else:
    print("\n[6b] Corriger les manquants ci-dessus avant de lancer Cellule 7.")


In [ ]:
# ══════ CELLULE 6c — PATCH PYTORCH3D ══════
# A lancer si Cell 7 plante sur : ModuleNotFoundError: No module named 'pytorch3d'
# Cree le shim pytorch3d directement dans ~/GVHMR (toujours dans PYTHONPATH subprocess).
# Compatible avec toute version de motus_extract — aucun reload requis.

import os

GVHMR_DIR = os.path.expanduser("~/GVHMR")
_shim = os.path.join(GVHMR_DIR, "pytorch3d")
_xforms = os.path.join(_shim, "transforms")

for _d in (_xforms,
           os.path.join(_shim, "structures"),
           os.path.join(_shim, "renderer")):
    os.makedirs(_d, exist_ok=True)

with open(os.path.join(_shim, "__init__.py"), "w") as _f:
    _f.write("# pytorch3d minimal shim — ANIMA-MECHANICUS\n")
for _sub in ("structures", "renderer"):
    open(os.path.join(_shim, _sub, "__init__.py"), "w").close()

_transforms_code = '''
import torch
import torch.nn.functional as F

def quaternion_to_matrix(q):
    r, i, j, k = torch.unbind(q, -1)
    s = 2.0 / (q * q).sum(-1)
    o = torch.stack([
        1 - s*(j*j + k*k), s*(i*j - k*r),     s*(i*k + j*r),
        s*(i*j + k*r),     1 - s*(i*i + k*k), s*(j*k - i*r),
        s*(i*k - j*r),     s*(j*k + i*r),     1 - s*(i*i + j*j),
    ], -1)
    return o.reshape(q.shape[:-1] + (3, 3))

def _sqrt_pos(x):
    return torch.where(x > 0, torch.sqrt(torch.clamp(x, min=0)), torch.zeros_like(x))

def matrix_to_quaternion(m):
    b = m.shape[:-2]
    t = torch.unbind(m.reshape(b + (9,)), -1)
    m00,m01,m02,m10,m11,m12,m20,m21,m22 = t
    q = _sqrt_pos(torch.stack([1+m00+m11+m22, 1+m00-m11-m22,
                               1-m00+m11-m22, 1-m00-m11+m22], -1) / 4.0)
    w, x, y, z = torch.unbind(q, -1)
    r = torch.stack([w, torch.copysign(x, m21-m12),
                     torch.copysign(y, m02-m20),
                     torch.copysign(z, m10-m01)], -1)
    return r / r.norm(dim=-1, keepdim=True).clamp(min=1e-8)

def axis_angle_to_quaternion(aa):
    ang = torch.norm(aa, p=2, dim=-1, keepdim=True).clamp(min=1e-6)
    h = ang * 0.5
    s = torch.where(ang < 1e-6, 0.5 - (ang*ang)/48, torch.sin(h)/ang)
    return torch.cat([torch.cos(h), aa * s], -1)

def axis_angle_to_matrix(aa):
    return quaternion_to_matrix(axis_angle_to_quaternion(aa))

def quaternion_to_axis_angle(q):
    n = torch.norm(q[..., 1:], p=2, dim=-1, keepdim=True)
    h = torch.atan2(n, q[..., :1])
    ang = 2 * h
    s = torch.where(ang < 1e-6, 0.5 - (ang*ang)/48, torch.sin(h)/ang)
    return q[..., 1:] / s.clamp(min=1e-8)

def matrix_to_axis_angle(m):
    return quaternion_to_axis_angle(matrix_to_quaternion(m))

def rotation_6d_to_matrix(d6):
    a1, a2 = d6[..., :3], d6[..., 3:]
    b1 = F.normalize(a1, dim=-1)
    b2 = F.normalize(a2 - (b1 * a2).sum(-1, keepdim=True) * b1, dim=-1)
    b3 = torch.linalg.cross(b1, b2)
    return torch.stack((b1, b2, b3), dim=-2)

def matrix_to_rotation_6d(m):
    return m[..., :2, :].clone().reshape(m.shape[:-2] + (6,))

def standardize_quaternion(q):
    return torch.where(q[..., 0:1] < 0, -q, q)

def quaternion_raw_multiply(a, b):
    aw, ax, ay, az = torch.unbind(a, -1)
    bw, bx, by, bz = torch.unbind(b, -1)
    return torch.stack([aw*bw-ax*bx-ay*by-az*bz,
                        aw*bx+ax*bw+ay*bz-az*by,
                        aw*by-ax*bz+ay*bw+az*bx,
                        aw*bz+ax*by-ay*bx+az*bw], -1)

def quaternion_multiply(a, b):
    return standardize_quaternion(quaternion_raw_multiply(a, b))

def quaternion_invert(q):
    s = torch.tensor([1, -1, -1, -1], dtype=q.dtype, device=q.device)
    return q * s
'''

with open(os.path.join(_xforms, "__init__.py"), "w") as _f:
    _f.write(_transforms_code)

# Forcer reload de motus_extract si deja en memoire
import sys, importlib
if "motus_extract" in sys.modules:
    importlib.reload(sys.modules["motus_extract"])
    print("[6c] motus_extract rechargé (cached version remplacee)")

print(f"[6c] shim pytorch3d cree dans {_shim}")
print("[6c] Pret — lancer Cell 7")


In [ ]:
# ══════ CELLULE 7 — EXTRACTION GVHMR → SMPL→R15 → .npz ══════
# Duree estimee : 1-3 min par segment de 30s sur Colab T4

import sys, os, tempfile
import numpy as np
import cv2

sys.path.insert(0, f"{REPO_DIR}/U-ALPHA/codebase")
from motus_extract import (
    run_gvhmr, smpl_to_r15, fill_gaps,
    smooth_array, smooth_extremities,
    temporal_resample, normalize_quats,
    export_npz, mask_lower_body_joints,
    SMOOTH_PRESETS
)

assert SEGMENTS_TO_PROCESS, "[ERREUR] Valider les segments en Cellule 6"

# Filtrer segments GVHMR (exclure DECA)
GVHMR_SEGMENTS = [s for s in SEGMENTS_TO_PROCESS if not s.get('_skip_gvhmr')]
SKIPPED_SEGMENTS = [s for s in SEGMENTS_TO_PROCESS if s.get('_skip_gvhmr')]

if not GVHMR_SEGMENTS:
    raise ValueError("[ERREUR] Aucun segment GVHMR — relancer Cellule 6")

if SKIPPED_SEGMENTS:
    print(f"[INFO] {len(SKIPPED_SEGMENTS)} segment(s) DECA ignores — non implemente")
    for s in SKIPPED_SEGMENTS:
        print(f"  P{s['person_id']} {s['start_s']:.1f}s→{s['end_s']:.1f}s | {s.get('modele_recommande','?')}")
    print()

# Info segments haut du corps (FrankMocap_upper → masquage joints inferieurs)
_frank_segs = [s for s in GVHMR_SEGMENTS if s.get('_mask_lower_body')]
if _frank_segs:
    print(f"[INFO] {len(_frank_segs)} segment(s) haut du corps — joints inferieurs masques (quaternion identite)")
    for s in _frank_segs:
        print(f"  P{s['person_id']} {s['start_s']:.1f}s→{s['end_s']:.1f}s")
    print()

cap = cv2.VideoCapture(VIDEO_PATH)
src_fps    = cap.get(cv2.CAP_PROP_FPS)
tot_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
duration   = tot_frames / src_fps
cap.release()
print(f"[U-ALPHA] Source : {src_fps:.1f} FPS, {duration:.1f}s")
print(f"[U-ALPHA] {len(GVHMR_SEGMENTS)} segment(s) GVHMR a traiter")

# Etape 1 : GVHMR
print("\n[U-ALPHA] 1/3 — Extraction GVHMR (poses SMPL monde)...")
with tempfile.TemporaryDirectory() as tmp_dir:
    gvhmr_tracks = run_gvhmr(VIDEO_PATH, GVHMR_SEGMENTS, GVHMR_DIR, tmp_dir)

if not gvhmr_tracks:
    raise RuntimeError("[ERREUR] GVHMR n'a produit aucun resultat")

# Etapes 2-3 : Retargeting + Lissage + Export
print("\n[U-ALPHA] 2/3 — Retargeting SMPL→R15 + Lissage...")
win, poly = SMOOTH_PRESETS[LISSAGE]
NPZ_FILES = []
n_persons = len(gvhmr_tracks)

for pid in sorted(gvhmr_tracks.keys()):
    track = gvhmr_tracks[pid]
    print(f"  Personne {pid} : {len(track['poses'])} frames brutes")

    fill_gaps(track["poses"], track["transl"])
    indices  = sorted(track["poses"].keys())
    poses_aa = np.array([track["poses"][i] for i in indices])
    transl   = np.array([track["transl"][i] for i in indices])

    rots, root_pos = smpl_to_r15(poses_aa, transl)

    # Masquage joints inferieurs pour segments haut du corps
    if track.get("mask_lower_body"):
        rots = mask_lower_body_joints(rots)

    if not ROOT_MOTION:
        root_pos = np.zeros_like(root_pos)

    root_pos = smooth_array(root_pos, win, poly)
    rots = smooth_array(rots.reshape(len(rots), -1), win, poly).reshape(-1, 15, 4)
    rots = normalize_quats(rots)
    rots = smooth_extremities(rots, max(win * 2 + 1, 15))

    root_pos = temporal_resample(root_pos, src_fps, FPS_CIBLE)
    rots = temporal_resample(rots.reshape(len(rots), -1), src_fps, FPS_CIBLE).reshape(-1, 15, 4)
    rots = normalize_quats(rots)

    # Validation avant sauvegarde
    assert rots.shape == (len(rots), 15, 4), f"Shape rotations incorrecte : {rots.shape}"
    assert not np.any(np.isnan(rots)), "NaN detecte dans les rotations"
    assert not np.any(np.isnan(root_pos)), "NaN detecte dans root_position"

    out_path = f"{OUTPUT_DIR}/motus_core_P{pid}.npz"
    export_npz(rots, root_pos, FPS_CIBLE, src_fps, duration, pid, n_persons, out_path)
    NPZ_FILES.append(out_path)
    size_kb = os.path.getsize(out_path) / 1024
    mask_info = " [haut_corps]" if track.get("mask_lower_body") else ""
    print(f"  → {os.path.basename(out_path)} | {rots.shape[0]} frames @ {FPS_CIBLE} FPS | {size_kb:.1f} KB{mask_info}")

print(f"\n[U-ALPHA] {len(NPZ_FILES)} .npz exportes — Passer a la Cellule 8")
if SKIPPED_SEGMENTS:
    print(f"[INFO]   {len(SKIPPED_SEGMENTS)} segment(s) DECA non traites")


In [ ]:
# ══════ CELLULE 8 — TELECHARGEMENT DES .npz ══════

from google.colab import files as colab_files
import numpy as np, os

assert NPZ_FILES, "[ERREUR] Aucun .npz genere — relancer la Cellule 7"

print(f"[U-ALPHA] {len(NPZ_FILES)} fichier(s) .npz prets :")
for npz_path in NPZ_FILES:
    d = np.load(npz_path, allow_pickle=True)
    size_kb = os.path.getsize(npz_path) / 1024
    print(f"  {os.path.basename(npz_path)} | {d['rotations'].shape[0]} frames | {size_kb:.1f} KB")
    print(f"    rotations    : {d['rotations'].shape} {d['rotations'].dtype}")
    print(f"    root_position: {d['root_position'].shape} {d['root_position'].dtype}")
    print(f"    fps          : {int(d['fps'])} | duration : {float(d['duration']):.1f}s")

print("\nTelechargement en cours...")
for npz_path in NPZ_FILES:
    colab_files.download(npz_path)

print("\n[U-ALPHA] Mission accomplie.")
print("Prochain pas : Ouvrir ANIMA_MECHANICUS_GAMMA.ipynb")
print("  → Uploader ces .npz dans la Fregate U-GAMMA (La Forge)")